# Medication data analysis (Snowflake, read-only)

One row = one medication entry. **Read-only**: `SELECT` / `DESCRIBE` only.

**Grain:** `MedicationId` should be unique. One patient / encounter can have many medications.

**Most important views (same idea as SNOMED + Value)**
- Unique **Medication Name** with occurrences, plus NDC codes attached to that name.
- Unique **NDC Code** with occurrences, plus names attached.
- **Same NDC, different names** (and the reverse: one name, several NDCs).
- `Medication Status`, `Route`, `Frequency`, `Dosage` as stored strings with counts.

`Medication Name`, `NDC Code`, `Quantity Dispensed`, `Days Supply`, `Medication Status`, and `Prescription Date` have spaces — they are quoted in config.

## 1. Active session

In [ ]:
import pandas as pd

from snowflake.snowpark.context import get_active_session

session = get_active_session()
session

## 2. Config (change names ONLY here)

In [ ]:
DATABASE_NAME = "ATTR"
SCHEMA_NAME = "PUBLIC"
TABLE_NAME = "MEDICATION"   # try MEDICATIONS, RX, DRUG, PRESCRIPTION

QUOTE_DATABASE = False
QUOTE_SCHEMA = False
QUOTE_TABLE = False
QUOTE_COLUMNS = True

COL = {
    "medication_id": "MedicationId",
    "encounter_id": "EncounterId/VisitId",
    "patient_id": "Member/PatientId",
    "prescriber_npi": "PrescribingProviderNPI",
    "medication_name": "Medication Name",
    "ndc_code": "NDC Code",
    "dosage": "Dosage",
    "frequency": "Frequency",
    "route": "Route",
    "quantity_dispensed": "Quantity Dispensed",
    "days_supply": "Days Supply",
    "medication_status": "Medication Status",
    "prescription_date": "Prescription Date",
    "date": "Date",
}


def sf_ident(name, quoted):
    if quoted:
        return '"' + str(name).replace('"', '""') + '"'
    return str(name)


def col(key):
    return sf_ident(COL[key], QUOTE_COLUMNS)


DB = sf_ident(DATABASE_NAME, QUOTE_DATABASE)
T = ".".join(
    [
        DB,
        sf_ident(SCHEMA_NAME, QUOTE_SCHEMA),
        sf_ident(TABLE_NAME, QUOTE_TABLE),
    ]
)

C_ID = col("medication_id")
C_ENC = col("encounter_id")
C_PT = col("patient_id")
C_NPI = col("prescriber_npi")
C_NAME = col("medication_name")
C_NDC = col("ndc_code")
C_DOSE = col("dosage")
C_FREQ = col("frequency")
C_ROUTE = col("route")
C_QTY = col("quantity_dispensed")
C_DAYS = col("days_supply")
C_STATUS = col("medication_status")
C_RXDATE = col("prescription_date")
C_DATE = col("date")

for name, value in [
    ("DB", DB),
    ("T", T),
    ("C_ID", C_ID),
    ("C_ENC", C_ENC),
    ("C_PT", C_PT),
    ("C_NPI", C_NPI),
    ("C_NAME", C_NAME),
    ("C_NDC", C_NDC),
    ("C_DOSE", C_DOSE),
    ("C_FREQ", C_FREQ),
    ("C_ROUTE", C_ROUTE),
    ("C_QTY", C_QTY),
    ("C_DAYS", C_DAYS),
    ("C_STATUS", C_STATUS),
    ("C_RXDATE", C_RXDATE),
    ("C_DATE", C_DATE),
]:
    print(f"{name} = {value}")

## 3. Find the table (only if the name or schema is wrong)

In [ ]:
SELECT
    CURRENT_ROLE() AS ROLE,
    CURRENT_WAREHOUSE() AS WAREHOUSE,
    CURRENT_DATABASE() AS DATABASE,
    CURRENT_SCHEMA() AS SCHEMA;

In [ ]:
SELECT
    TABLE_CATALOG,
    TABLE_SCHEMA,
    TABLE_NAME,
    ROW_COUNT,
    BYTES
FROM {{DB}}.INFORMATION_SCHEMA.TABLES
WHERE TABLE_TYPE = 'BASE TABLE'
  AND (
        UPPER(TABLE_NAME) LIKE '%MEDICATION%'
     OR UPPER(TABLE_NAME) LIKE '%DRUG%'
     OR UPPER(TABLE_NAME) LIKE '%RX%'
     OR UPPER(TABLE_NAME) LIKE '%PRESCRIB%'
  )
ORDER BY TABLE_SCHEMA, TABLE_NAME;

## 4. Table shape and first 10 rows

In [ ]:
DESCRIBE TABLE {{T}};

In [ ]:
SELECT *
FROM {{T}}
LIMIT 10;

## 5. Volume and uniqueness

In [ ]:
SELECT
    COUNT(*) AS ROW_COUNT,
    COUNT(DISTINCT {{C_ID}}) AS UNIQUE_MEDICATION_IDS,
    COUNT(DISTINCT {{C_PT}}) AS UNIQUE_PATIENTS,
    COUNT(DISTINCT {{C_ENC}}) AS UNIQUE_ENCOUNTERS,
    COUNT(DISTINCT {{C_NAME}}) AS UNIQUE_MEDICATION_NAMES,
    COUNT(DISTINCT {{C_NDC}}) AS UNIQUE_NDC_CODES,
    COUNT(DISTINCT {{C_NPI}}) AS UNIQUE_PRESCRIBER_NPIS,
    COUNT(*) - COUNT(DISTINCT {{C_ID}}) AS EXTRA_ROWS_VS_UNIQUE_ID,
    ROUND(COUNT(*) / NULLIF(COUNT(DISTINCT {{C_PT}}), 0), 2) AS AVG_MEDS_PER_PATIENT
FROM {{T}};

## 6. Completeness (nulls)

Required: MedicationId, encounter, patient, Medication Name, Medication Status, Prescription Date, Date. NDC, dosage, frequency, route, quantity, days supply, prescriber NPI are optional.

In [ ]:
SELECT
    COUNT(*) AS ROW_COUNT,
    SUM(IFF({{C_ID}} IS NULL, 1, 0)) AS NULL_MEDICATION_ID,
    SUM(IFF({{C_ENC}} IS NULL, 1, 0)) AS NULL_ENCOUNTER_ID,
    SUM(IFF({{C_PT}} IS NULL, 1, 0)) AS NULL_PATIENT_ID,
    SUM(IFF({{C_NPI}} IS NULL, 1, 0)) AS NULL_PRESCRIBER_NPI,
    SUM(IFF({{C_NAME}} IS NULL, 1, 0)) AS NULL_MEDICATION_NAME,
    SUM(IFF({{C_NDC}} IS NULL, 1, 0)) AS NULL_NDC_CODE,
    SUM(IFF({{C_DOSE}} IS NULL, 1, 0)) AS NULL_DOSAGE,
    SUM(IFF({{C_FREQ}} IS NULL, 1, 0)) AS NULL_FREQUENCY,
    SUM(IFF({{C_ROUTE}} IS NULL, 1, 0)) AS NULL_ROUTE,
    SUM(IFF({{C_QTY}} IS NULL, 1, 0)) AS NULL_QUANTITY_DISPENSED,
    SUM(IFF({{C_DAYS}} IS NULL, 1, 0)) AS NULL_DAYS_SUPPLY,
    SUM(IFF({{C_STATUS}} IS NULL, 1, 0)) AS NULL_MEDICATION_STATUS,
    SUM(IFF({{C_RXDATE}} IS NULL, 1, 0)) AS NULL_PRESCRIPTION_DATE,
    SUM(IFF({{C_DATE}} IS NULL, 1, 0)) AS NULL_DATE
FROM {{T}};

## 7. Medication Name — unique names with occurrences (and NDC codes)

One row per stored name. `ROW_COUNT` is how often that spelling appears. `NDC_CODES` lists the NDC values attached to that name.

In [ ]:
SELECT
    {{C_NAME}} AS MEDICATION_NAME,
    COUNT(*) AS ROW_COUNT,
    COUNT(DISTINCT {{C_PT}}) AS UNIQUE_PATIENTS,
    COUNT(DISTINCT {{C_NDC}}) AS DISTINCT_NDC_CODES,
    LISTAGG(DISTINCT {{C_NDC}}::STRING, ' | ') AS NDC_CODES,
    COUNT(DISTINCT {{C_STATUS}}) AS DISTINCT_STATUSES,
    ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS PCT_OF_ROWS
FROM {{T}}
GROUP BY 1
ORDER BY ROW_COUNT DESC, MEDICATION_NAME;

### Word counts inside Medication Name

`Metformin HCl` is one name; the word `Metformin` can appear in several names. This is a short field, so word split is usually cheap (unlike 40M clinical notes).

In [ ]:
WITH tokens AS (
    SELECT
        {{C_PT}} AS PATIENT_ID,
        TRIM(f.VALUE::STRING) AS WORD
    FROM {{T}},
         LATERAL FLATTEN(
             INPUT => SPLIT(
                 TRIM(REGEXP_REPLACE({{C_NAME}}::STRING, '[^A-Za-z0-9]+', ' ')),
                 ' '
             )
         ) f
    WHERE {{C_NAME}} IS NOT NULL
)
SELECT
    WORD,
    COUNT(*) AS WORD_OCCURRENCES,
    COUNT(DISTINCT PATIENT_ID) AS UNIQUE_PATIENTS,
    ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS PCT_OF_WORD_OCCURRENCES
FROM tokens
WHERE WORD IS NOT NULL
  AND WORD <> ''
GROUP BY 1
ORDER BY WORD_OCCURRENCES DESC, WORD;

## 8. NDC Code — unique codes, names, same code different names

NDC identifies the product. The same NDC should usually have one name; several names for one NDC is a data-quality issue (same idea as one SNOMED, several Value spellings).

In [ ]:
SELECT
    COUNT(*) AS ROW_COUNT,
    COUNT(DISTINCT NULLIF(TRIM({{C_NDC}}::STRING), '')) AS UNIQUE_NDC_CODES,
    SUM(IFF({{C_NDC}} IS NOT NULL AND TRIM({{C_NDC}}::STRING) <> '', 1, 0)) AS ROWS_WITH_NDC,
    SUM(IFF({{C_NDC}} IS NULL OR TRIM({{C_NDC}}::STRING) = '', 1, 0)) AS ROWS_MISSING_NDC,
    ROUND(
        100.0 * SUM(IFF({{C_NDC}} IS NOT NULL AND TRIM({{C_NDC}}::STRING) <> '', 1, 0))
        / NULLIF(COUNT(*), 0),
        2
    ) AS PCT_ROWS_WITH_NDC
FROM {{T}};

In [ ]:
SELECT
    {{C_NDC}} AS NDC_CODE,
    {{C_NAME}} AS MEDICATION_NAME,
    COUNT(*) AS ROW_COUNT,
    COUNT(DISTINCT {{C_PT}}) AS UNIQUE_PATIENTS,
    COUNT(*) OVER (PARTITION BY {{C_NDC}}) AS NAMES_FOR_THIS_NDC,
    ROW_NUMBER() OVER (PARTITION BY {{C_NDC}} ORDER BY COUNT(*) DESC) AS NAME_NUMBER,
    ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (PARTITION BY {{C_NDC}}), 2) AS PCT_WITHIN_NDC
FROM {{T}}
WHERE {{C_NDC}} IS NOT NULL
  AND TRIM({{C_NDC}}::STRING) <> ''
GROUP BY 1, 2
ORDER BY NAMES_FOR_THIS_NDC DESC, NDC_CODE, ROW_COUNT DESC;

In [ ]:
SELECT
    {{C_NDC}} AS NDC_CODE,
    COUNT(*) AS ROW_COUNT,
    COUNT(DISTINCT {{C_NAME}}) AS UNIQUE_NAMES,
    COUNT(DISTINCT UPPER(REGEXP_REPLACE(TRIM({{C_NAME}}::STRING), '\\s+', ' '))) AS UNIQUE_NAMES_NORMALIZED,
    LISTAGG(DISTINCT {{C_NAME}}::STRING, ' | ') AS NAME_VARIATIONS
FROM {{T}}
WHERE {{C_NDC}} IS NOT NULL
  AND TRIM({{C_NDC}}::STRING) <> ''
GROUP BY 1
ORDER BY UNIQUE_NAMES DESC, ROW_COUNT DESC;

In [ ]:
SELECT
    {{C_NDC}} AS NDC_CODE,
    COUNT(*) AS ROW_COUNT,
    COUNT(DISTINCT {{C_NAME}}) AS UNIQUE_NAMES,
    LISTAGG(DISTINCT {{C_NAME}}::STRING, ' | ') AS NAME_VARIATIONS
FROM {{T}}
WHERE {{C_NDC}} IS NOT NULL
  AND TRIM({{C_NDC}}::STRING) <> ''
GROUP BY 1
HAVING COUNT(DISTINCT {{C_NAME}}) > 1
ORDER BY UNIQUE_NAMES DESC, ROW_COUNT DESC;

In [ ]:
SELECT
    {{C_NAME}} AS MEDICATION_NAME,
    COUNT(*) AS ROW_COUNT,
    COUNT(DISTINCT {{C_NDC}}) AS UNIQUE_NDC_CODES,
    LISTAGG(DISTINCT {{C_NDC}}::STRING, ' | ') AS NDC_CODES
FROM {{T}}
WHERE {{C_NAME}} IS NOT NULL
GROUP BY 1
HAVING COUNT(DISTINCT {{C_NDC}}) > 1
ORDER BY UNIQUE_NDC_CODES DESC, ROW_COUNT DESC;

## 9. Status, Route, Frequency, Dosage

Dictionary status examples: Active, Discontinued, Completed, On Hold. We group by the **stored string** (spelling variants stay separate).

In [ ]:
SELECT
    {{C_STATUS}} AS MEDICATION_STATUS,
    COUNT(*) AS ROW_COUNT,
    COUNT(DISTINCT {{C_PT}}) AS UNIQUE_PATIENTS,
    COUNT(DISTINCT {{C_NAME}}) AS UNIQUE_MEDICATION_NAMES,
    ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS PCT_OF_ROWS
FROM {{T}}
GROUP BY 1
ORDER BY ROW_COUNT DESC;

In [ ]:
SELECT
    {{C_ROUTE}} AS ROUTE,
    COUNT(*) AS ROW_COUNT,
    COUNT(DISTINCT {{C_PT}}) AS UNIQUE_PATIENTS,
    COUNT(DISTINCT {{C_NAME}}) AS UNIQUE_MEDICATION_NAMES,
    ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS PCT_OF_ROWS
FROM {{T}}
GROUP BY 1
ORDER BY ROW_COUNT DESC;

In [ ]:
SELECT
    {{C_FREQ}} AS FREQUENCY,
    COUNT(*) AS ROW_COUNT,
    COUNT(DISTINCT {{C_PT}}) AS UNIQUE_PATIENTS,
    ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS PCT_OF_ROWS
FROM {{T}}
GROUP BY 1
ORDER BY ROW_COUNT DESC;

In [ ]:
SELECT
    {{C_DOSE}} AS DOSAGE,
    COUNT(*) AS ROW_COUNT,
    COUNT(DISTINCT {{C_NAME}}) AS UNIQUE_MEDICATION_NAMES,
    ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS PCT_OF_ROWS
FROM {{T}}
GROUP BY 1
ORDER BY ROW_COUNT DESC;

In [ ]:
SELECT
    {{C_NAME}} AS MEDICATION_NAME,
    {{C_DOSE}} AS DOSAGE,
    {{C_FREQ}} AS FREQUENCY,
    {{C_ROUTE}} AS ROUTE,
    COUNT(*) AS ROW_COUNT,
    COUNT(DISTINCT {{C_PT}}) AS UNIQUE_PATIENTS
FROM {{T}}
GROUP BY 1, 2, 3, 4
ORDER BY ROW_COUNT DESC
LIMIT 200;

## 10. Prescribing provider NPI

In [ ]:
SELECT
    {{C_NPI}} AS PRESCRIBING_PROVIDER_NPI,
    COUNT(*) AS ROW_COUNT,
    COUNT(DISTINCT {{C_PT}}) AS UNIQUE_PATIENTS,
    COUNT(DISTINCT {{C_NAME}}) AS UNIQUE_MEDICATION_NAMES
FROM {{T}}
WHERE {{C_NPI}} IS NOT NULL
GROUP BY 1
ORDER BY ROW_COUNT DESC
LIMIT 50;

## 11. Prescription Date and recorded Date

`Prescription Date` = when the Rx was written. `Date` = when this row was recorded. Recency uses **Prescription Date** vs today.

In [ ]:
SELECT
    CURRENT_DATE() AS TODAY,
    MIN({{C_RXDATE}}) AS MIN_PRESCRIPTION_DATE,
    MAX({{C_RXDATE}}) AS MAX_PRESCRIPTION_DATE,
    MIN({{C_DATE}}) AS MIN_RECORDED_DATE,
    MAX({{C_DATE}}) AS MAX_RECORDED_DATE,
    SUM(IFF({{C_RXDATE}}::DATE > CURRENT_DATE(), 1, 0)) AS FUTURE_PRESCRIPTION_DATES,
    SUM(IFF({{C_RXDATE}}::DATE IS DISTINCT FROM {{C_DATE}}::DATE, 1, 0)) AS ROWS_RX_DATE_NE_RECORDED_DATE
FROM {{T}};

In [ ]:
SELECT
    YEAR({{C_RXDATE}}) AS PRESCRIPTION_YEAR,
    COUNT(*) AS ROW_COUNT,
    COUNT(DISTINCT {{C_PT}}) AS UNIQUE_PATIENTS,
    COUNT(DISTINCT {{C_NAME}}) AS UNIQUE_MEDICATION_NAMES,
    ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS PCT_OF_ROWS
FROM {{T}}
GROUP BY 1
ORDER BY 1;

In [ ]:
WITH bucketed AS (
    SELECT
        CASE
            WHEN {{C_RXDATE}} IS NULL THEN 90
            WHEN {{C_RXDATE}}::DATE > CURRENT_DATE() THEN 80
            WHEN DATEDIFF('day', {{C_RXDATE}}::DATE, CURRENT_DATE()) <= 30 THEN 1
            WHEN DATEDIFF('day', {{C_RXDATE}}::DATE, CURRENT_DATE()) <= 90 THEN 2
            WHEN DATEDIFF('day', {{C_RXDATE}}::DATE, CURRENT_DATE()) <= 180 THEN 3
            WHEN DATEDIFF('day', {{C_RXDATE}}::DATE, CURRENT_DATE()) <= 365 THEN 4
            WHEN DATEDIFF('year', {{C_RXDATE}}::DATE, CURRENT_DATE()) <= 2 THEN 5
            WHEN DATEDIFF('year', {{C_RXDATE}}::DATE, CURRENT_DATE()) <= 5 THEN 6
            WHEN DATEDIFF('year', {{C_RXDATE}}::DATE, CURRENT_DATE()) <= 7 THEN 7
            WHEN DATEDIFF('year', {{C_RXDATE}}::DATE, CURRENT_DATE()) <= 10 THEN 8
            ELSE 9
        END AS SORT_ORDER,
        {{C_PT}} AS PATIENT_ID
    FROM {{T}}
)
SELECT
    SORT_ORDER,
    CASE SORT_ORDER
        WHEN 1 THEN '0-30 days ago'
        WHEN 2 THEN '31-90 days ago'
        WHEN 3 THEN '91-180 days ago'
        WHEN 4 THEN '181-365 days ago'
        WHEN 5 THEN '1-2 years ago'
        WHEN 6 THEN '3-5 years ago'
        WHEN 7 THEN '6-7 years ago'
        WHEN 8 THEN '8-10 years ago'
        WHEN 9 THEN 'More than 10 years ago'
        WHEN 80 THEN 'Future (after today)'
        ELSE 'Unknown (missing date)'
    END AS DATE_RANGE,
    COUNT(*) AS ROW_COUNT,
    COUNT(DISTINCT PATIENT_ID) AS UNIQUE_PATIENTS,
    ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS PCT_OF_ROWS
FROM bucketed
GROUP BY 1, 2
ORDER BY SORT_ORDER;

## 12. Medications per patient and one patient drill-down

In [ ]:
WITH per_pt AS (
    SELECT
        {{C_PT}} AS PATIENT_ID,
        COUNT(*) AS ROW_COUNT
    FROM {{T}}
    WHERE {{C_PT}} IS NOT NULL
    GROUP BY 1
)
SELECT
    ROW_COUNT AS MEDS_PER_PATIENT,
    COUNT(*) AS NUMBER_OF_PATIENTS,
    ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS PCT_OF_PATIENTS
FROM per_pt
GROUP BY 1
ORDER BY 1;

In [ ]:
SELECT
    {{C_PT}} AS PATIENT_ID,
    COUNT(*) AS ROW_COUNT,
    COUNT(DISTINCT {{C_NAME}}) AS UNIQUE_MEDICATION_NAMES,
    COUNT(DISTINCT {{C_NDC}}) AS UNIQUE_NDC_CODES,
    COUNT(DISTINCT {{C_STATUS}}) AS DISTINCT_STATUSES,
    MIN({{C_RXDATE}}) AS FIRST_RX_DATE,
    MAX({{C_RXDATE}}) AS LAST_RX_DATE
FROM {{T}}
WHERE {{C_PT}} IS NOT NULL
GROUP BY 1
ORDER BY ROW_COUNT DESC
LIMIT 25;

In [ ]:
SELECT
    {{C_PT}} AS PATIENT_ID,
    {{C_ID}} AS MEDICATION_ID,
    {{C_ENC}} AS ENCOUNTER_ID,
    {{C_NAME}} AS MEDICATION_NAME,
    {{C_NDC}} AS NDC_CODE,
    {{C_DOSE}} AS DOSAGE,
    {{C_FREQ}} AS FREQUENCY,
    {{C_ROUTE}} AS ROUTE,
    {{C_STATUS}} AS MEDICATION_STATUS,
    {{C_RXDATE}} AS PRESCRIPTION_DATE,
    {{C_DATE}} AS RECORDED_DATE
FROM {{T}}
WHERE {{C_PT}} = (
        SELECT {{C_PT}}
        FROM {{T}}
        WHERE {{C_PT}} IS NOT NULL
        GROUP BY 1
        ORDER BY COUNT(*) DESC
        LIMIT 1
      )
ORDER BY {{C_RXDATE}} NULLS LAST, {{C_ID}};

## Notes

- **Downloading:** use the download arrow on each SQL result grid.
- If `LISTAGG` of NDC lists for one name fails (huge list), use `ndc_name_pairs` instead — one row per NDC + name.
- Run `config` before SQL cells.